# INIT

In [ ]:
#!pip install PyPDF2
!pip install pymupdf
!pip install fitz
!pip install tools
!pip install src

  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for src
  Running setup.py clean for src
Failed to build src
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (src)


In [ ]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.6/201.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 6.8 MB/s eta 

In [ ]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=120ce99ce13ae5dd8af7e4ca6fdf966afbd3f1b49b461efb217300690e5d95a5
  Stored in directory: /root/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect


In [ ]:
#Modules and imports
import re
import os
import fitz
import pymupdf
import requests
import json
import pandas as pd
import torch
from pathlib import Path

from sentence_transformers import SentenceTransformer
from urllib.parse import urlparse
from typing import List, Dict
from transformers import AutoTokenizer

from chromadb import PersistentClient
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
import tiktoken
from langdetect import detect
#from PyPDF2 import PdfReader as pdr

In [ ]:
import nltk
nltk.download('punkt')  # only needs to be run once
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
from google.colab import drive
drive.mount('/content/gdrive')
from functools import reduce
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
  print("Device:", torch.cuda.get_device_name(0))
from hashlib import sha256

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
CUDA available: False


In [ ]:
#Pathing
path = "/content/gdrive/MyDrive/ColabNotebooks/DATACLEANING/dogman"
#path = "/content/gdrive/MyDrive/Q A Group/extracted_texts"
dataPath = f"{path}/data"
savePath = f"{dataPath}/JSONs"
pdfPath = f"{dataPath}/pdfs"

model = SentenceTransformer("intfloat/multilingual-e5-base")

# Functions

In [ ]:
#Functions
def clean_text(text):
    text = re.sub(r"[^\x20-\x7E\n]", "", text)  # remove non-printable
    text = re.sub(r"\s+", " ", text)            # collapse spaces
    text = re.sub(r"\n+", "\n", text)           # collapse multiple newlines
    return text.strip()

# Function — sentence-aware chunker
def chunk_by_sentences(text, max_chars=1000):
    """Function for chunking text based off whole sentences.
       When chunk is about to have a sentence cut off have way through, ends current chunk
       and creates new chunk starting off with the previous sentence."""
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = ""
    idx = 0

    for sentence in sentences:
        if len(current_chunk) + len(sentence) + 1 <= max_chars:
            current_chunk += " " + sentence
        else:
            chunks.append({
                "id": idx,
                "heading": None,
                "text": current_chunk.strip()
            })
            idx += 1
            current_chunk = sentence
    if current_chunk:
        chunks.append({
            "id": idx,
            "heading": None,
            "text": current_chunk.strip()
        })
    return chunks

def extractText(fullString, startDelimiter, endDelimiter):
    startDelimiter = reduceDelimiter(startDelimiter)
    endDelimiter = reduceDelimiter(endDelimiter)
    startIndex = fullString.find(startDelimiter)
    contentStart = startIndex + len(startDelimiter)
    endIndex = fullString.find(endDelimiter, contentStart)
    if startIndex == -1 and endIndex != -1:
        return "No Start Delimiter found: " + startDelimiter
    elif endIndex == -1 and startIndex != -1:
        return fullString[contentStart:len(fullString)]
    elif startIndex == -1 and endIndex == -1:
        return "Neither Delimiter found: " + startDelimiter + " : " + endDelimiter
    return fullString[contentStart:endIndex]

def reduceDelimiter(delimiter):
    match = re.match(r'(\S+)(?:\s+(\S+))?', delimiter)
    if match:
        word1 = match.group(1)
        word2 = match.group(2)
        if word2:
            return f"{word1} {word2}"
        else:
            return word1
    return ""

def shorten(link):
    parsed_url = urlparse(link)
    filename = os.path.basename(parsed_url.path)
    return filename

def open_pdf_from_url(url):
    try:
        response = requests.get(url)
        response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)
        doc = pymupdf.open(stream=response.content, filetype="pdf")
        return doc
    except requests.exceptions.RequestException as e:
        print(f"Error downloading PDF from URL: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

In [ ]:
#amari's repurposed chunking code function
def singlePDFChunker(billArr, countryCode, pdfLink):
    billDoc = open_pdf_from_url(billArr[2])                          #grab file from path and store it as a PDF within the program memory
    if billDoc:
      fullText = ""
      pages = []
      for page in billDoc:
        tempText = clean_text(page.get_text())
        fullText += tempText
      #Amari's code from PDFChunkerFinal
      # Runs chunking function
      chunks = chunk_by_sentences(fullText)
      dupChunks = chunks
      # Prepare texts for embedding
      texts = [chunk['text'] for chunk in chunks]
      # Compute embeddings
      embeddings = model.encode(texts, show_progress_bar=True)
      # Combine text chunks and embeddings
      for chunk, emb in zip(chunks, embeddings):
        chunk["embedding"] = emb.tolist()

      combined_json_path = f"{path}/{countryCode}_{billArr[3]}_chunksWembeddings.json"
      with open(combined_json_path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)



In [ ]:
#Country Code, CSV title, Link, pdf File Name
for i in range(0,len(failCSV)):
  #for every bill
  codeArr = [failCSV.loc[i,"country"], failCSV.loc[i,"title"], failCSV.loc[i,"url"], shorten(failCSV.loc[i,"url"])]
  csvTitle  = failCSV.loc[i,"title"]
  countryCode = failCSV.loc[i,"country"]
  Link = failCSV.loc[i,"url"]
  billName = shorten(Link)
  singlePDFChunker(codeArr, countryCode, Link)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Batches: 0it [00:00, ?it/s]

Batches: 0it [00:00, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#PDF BOOKMARKS FUNCTION
def singleCountryJSON(csvOBJ, countryCode, pdfLink):
  billOBJ = []
  billIndex = 0
  #iterate through each pdf in csvOBJ array

  for B in csvOBJ:
    billDoc = open_pdf_from_url(B[2])                          #grab file from path and store it as a PDF within the program memory
    if billDoc:
      toc = billDoc.get_toc()
      #toc stands for table of contents. organized as a list of tuples that has three atributes, the first being the level the separation sits, and the second being the title, the third being the start index
      #initialize the structure of the highest level of the bill structure in a temporarilly assigned variable as to not cause overwrites
      tempBillOBJ = [{
          "BILLPDF" : B[3],
          "BILLTITLE" : B[1],
          "BILLNUM": billIndex,
          "HEADINGS" : []
        }
      ]
      tempEmbBillOBJ = [{
          "BILLPDF" : B[3],
          "BILLTITLE" : B[1],
          "BILLNUM": billIndex,
          "HEADINGS" : []
        }
      ]
      #these values need to be initialized and cleared before iterating through the PDF contents
      headingName = ""
      separatorName = ""
      textContent = ""
      textPrompt = ""
      fullText = ""
      headerCount = 0
      separatorCount = 0
      removeClean = []
      pages = []


      #this loop iterates through our table of contents and stores which table of content layers are under more than one sub-chapter/separator
      for x in range(len(toc)-1):
        if toc[x][0] != 0 and toc[x][0] != 1 and toc[x][0] != 2:                  #hypothetically zero wouldn't exist here but I included it anyways
          removeClean.append(x)

      #reverse the stored list to cleanly pop them off the table of contents
      removeClean.reverse()
      for rem in removeClean:
        toc.pop(rem)

      fullText = ""
      pageCount = 1

      for page in billDoc:
        tempText = clean_text(page.get_text())
        fullText += tempText
        pages.append(tempText)
        pageCount += 1


      for y in range(len(toc)-1):
        if toc[y][0] == 2:
          startDel = toc[y][1]
          check = 0
          count = 0
          while check == 0 and count < 1000:
            if toc[y+1][0] < 4:
              endDel = toc[y+1][1]
              check = 1
            count+=1

          textPrompt = ""
          promptLoc = toc[y][2]
          for L in range(promptLoc-1,len(pages)):
            textPrompt += pages[L]

          if y + 1 == len(toc)-1:
            textContent = extractText(textPrompt,startDel,"HOPEFULLY NONE OF OUR DOCUMENTS HAVE THIS EXACT LINE OF TEXT")
          else:
            textContent = extractText(textPrompt,startDel,endDel)
        """print(f"{y} : {len(toc)-1}")"""
        if toc[y][0] == 1:
          tempBill = tempBillOBJ[0]
          tempHead = tempBill["HEADINGS"]
          tempHead.append({
            "HEADINGNUM": headerCount,
            "HEADINGNAME" : toc[y][1],
            "SEPARATORS" : []
          })
          headerCount += 1
          separatorCount = 0
        elif toc[y][0] == 2:
          separatorName = toc[y][1]
          pageIndex = toc[y][2]
          tempBill = tempBillOBJ[0]
          tempHead = tempBill["HEADINGS"]
          tempSep = tempHead[headerCount-1]
          tempSep = tempSep["SEPARATORS"]
          tempSep.append({
                "SEPARATORNUM": separatorCount,
                "PAGEINDEX" : pageIndex,
                "SEPARATORTITLE": toc[y][1],
                "TEXTBLOCK": textContent
              })
          separatorCount += 1
      billOBJ.append(tempBillOBJ)
      billDoc.close()
      billIndex += 1

  try:
    os.makedirs(f"{savePath}/{countryCode}", exist_ok=True)
    print(f"Folder '{countryCode}' created successfully or already exists.")
  except OSError as e:
    print(f"Error creating folder '{countryCode}': {e}")

  with open(f"{savePath}/{countryCode}/{countryCode}Text.json",'w') as f:
    json.dump(billOBJ,f,indent=4)
  billOBJ = json.dumps(billOBJ,indent=4)
  print(billOBJ)


# Get PDF's / CSV's




In [ ]:
#Get OG csv
oldCSVPath = f"{path}/upov_docs_metadata.csv"
cachedJSONExistsCSVPath = f"{path}/existing_jsons.csv"
extraInclusionsCSV = f"{path}/inclusions.csv"

upov = pd.read_csv(oldCSVPath)
#cache = pd.read_csv(cachedJSONExistsCSVPath)
#inclusions = pd.read_csv(extraInclusionsCSV)

#Create new csv with info of the PDF's that need to be used in operation
#IDs of all the PDFs we are excluding / used as final exclusion
exclusionByFULLDISCARD = [
    10945,    #"hu083en.pdf"
    10522,    #"us219en.pdf"
    10577,    #"us220en.pdf"      # table
    10901,    #"za068en.pdf"      # (last page)
    19114,    #"ca225en.pdf"
    19159,10522     #"us219en-related.pdf"
]

#duplicates will be put in the same folder under a different JSON
duplicateListByID = [
    18594,    #"au320en.pdf"
    14960,    #"au315en.pdf"
    11675,    #"au380en.pdf"
    10966,    #"au503en.pdf"
    9526
]

currentFail = [
    22412,
    9520,
    9514,
    18977,
    9528,
    499,
    21005,
    21006,
    9521,
    15598,
    5221,
    5222,
    854,
    14722,
    9516,
    964,
    5661,
    1217,
    1243,
    1297,
    19152,
    1391,
    11025,
    9522,
    5991,
    5993,
    5994,
    5995,
    5996,
    5999,
    10910,
    10937,
    11815,    #"ie181en.pdf"      # all Ireland are weird VVV
    11811,    #"ie177en.pdf"
    11813,    #"ie179en.pdf"
    11814,    #"ie180en.pdf"
    2337,     #"ie075en.pdf"
    16028,    #"ke027en.pdf"      # no bkmk
    9524,
    21352,
    6977,
    16028,
    16029,
    13731,
    2859,
    7180,
    2951,
    9543,
    15323,
    16200,
    15544,
    11026,
    3082,
    12842,
    9525,
    9530,
    22410,
    22411,
    3304,
    10927,
    3591,
    3663,
    15391,
    15398,
    5303,
    9541,
    7491,
    3902,
    3898,
    16192,
    16191,
    9532,
    5426,
    10999,
    10522,
    16630,
    9535,
    21074,
    6566,
    13110,
    12011,
    6127,
    9536,
    10901
]
#exclude non english documents for now
EnglishList = [
    "English"
]
justExcludeWTO = [
    62
]


mask = ~upov["Unnamed: 0"].isin(justExcludeWTO)
failCSV = upov[mask]
usableCSV = upov[mask]

mask = ~upov["id"].isin(exclusionByFULLDISCARD)
usableCSV = usableCSV[mask]

mask = ~upov["id"].isin(duplicateListByID)
usableCSV = usableCSV[mask]

mask = upov["id"].isin(currentFail)
failCSV = usableCSV[mask]

mask = ~upov["id"].isin(currentFail)
usableCSV = usableCSV[mask]

"""mask = mask["language"].isin(EnglishList)
usableCSV = usableCSV[mask]
failCSV = failCSV[mask]"""

usableCSV = usableCSV.reset_index()
failCSV = failCSV.reset_index()
#output should be a new csv that program can operate on

/tmp/ipython-input-2647239316.py:130: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  usableCSV = usableCSV[mask]
/tmp/ipython-input-2647239316.py:133: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  usableCSV = usableCSV[mask]
/tmp/ipython-input-2647239316.py:136: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  failCSV = usableCSV[mask]
/tmp/ipython-input-2647239316.py:139: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  usableCSV = usableCSV[mask]




```
Unnamed: 0                                                             0
country                                                               AL
id                                                                    50
code                                                                 NaN
title                  Law No. 8880 dated 15.04.2002 on Plant Breeder...
titleDir                                                             ltr
shortTitle                                                           NaN
shortTitleDir                                                        NaN
yearOfVersion                                                     2002.0
cntryOrgCode                                                         NaN
cntryOrgTitle                                                        NaN
cntryOrgType                                                         NaN
subjectMatters                                                       NaN
subjectMatterTitles                                                  NaN
typeOfTexts                                                          NaN
typeOfTextTitles                                                     NaN
adoptedDate                                                          NaN
entryIntoForce                                                       NaN
judgmentDate                                                         NaN
dates                                                                NaN
superseded                                                         False
repealed                                                           False
meta                                                                 NaN
textOrder                                                            NaN
url                    https://wipolex-resources-eu-central-1-3589224...
language                                                         English
parts                                                                NaN
mimeTypes              [{'id': 209067, 'value': 'pdf', 'fileUrl': None}]
customTitle                                                         True
filename                                                     al031en.pdf
Name: 0, dtype: object```



GET PDFS

In [ ]:
#Country Code, CSV title, Link, pdf File Name
codeArr = [[usableCSV.loc[0,"country"], usableCSV.loc[0,"title"], usableCSV.loc[0,"url"], shorten(usableCSV.loc[0,"url"])]]


for i in range(1,len(usableCSV)):
  #for every bill
  csvTitle  = usableCSV.loc[i,"title"]
  countryCode = usableCSV.loc[i,"country"]
  Link = usableCSV.loc[i,"url"]
  billName = shorten(Link)

  if countryCode == codeArr[-1][0]:
    codeArr.append([countryCode,csvTitle,Link,billName])
  else:
    singleCountryJSON(codeArr, countryCode, Link)
    #print(codeArr)
    codeArr.clear()
    codeArr.append([countryCode,csvTitle,Link,billName])
#print(codeArr)
singleCountryJSON(codeArr, countryCode, Link)

# Jibek Code

In [ ]:
#config
HEADING_REGEX = re.compile(
    r"(?:^|\n)(?P<heading>(?:PART|CHAPTER|TITLE|SECTION|ANNEX)\s+[\w\dIVXLCDM\-\.]+[^\n]*)",
    re.IGNORECASE
)

ARTICLE_HEADING_REGEX = re.compile(
    r"(?:(?:^|\n)\s*)(?P<article>Article\s+(?P<number>\d{1,3}))\s*(?:\n|$)",
    flags=re.IGNORECASE
)

CHUNK_SIZE = 512

# EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"

EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-large"
LLM_NAME = "microsoft/Phi-4-mini-instruct"
#"MarsGray/Phi-4-mini-instruct-merged-1.1"
#
#"google/gemma-7b-it"
#"MarsGray/Phi-4-mini-instruct-merged"
#
#
RAW_PDF_DIR = "data/albania.pdf"
JSONL_OUTPUT_DIR = "processed/jsonl"
VECTOR_STORE_PATH = "processed/vector_store/"

In [ ]:
#section_splitter
def split_into_section(text):
    sections = []
    hierarchy = {
        "PART": None,
        "CHAPTER": None,
        "TITLE": None,
        "SECTION": None,
        "ANNEX": None
    }

    matches = []
    for m in HEADING_REGEX.finditer(text):
        matches.append({
            "type": "structure",
            "match": m,
            "start": m.start(),
            "end": m.end()
        })
    for m in ARTICLE_HEADING_REGEX.finditer(text):
        matches.append({
            "type": "article",
            "match": m,
            "start": m.start(),
            "end": m.end(),
            "article_number": int(m.group("number"))
        })

    matches.sort(key=lambda x: x["start"])

    # 1. Preamble
    if matches and matches[0]["start"] > 0:
        preamble = text[:matches[0]["start"]].strip()
        if preamble:
            sections.append({
                "heading": "PREAMBLE",
                "text": preamble,
                "sub_heading": []
            })

    # 2. Loop through matches and build sections
    i = 0
    while i < len(matches):
        m = matches[i]
        match_obj = m["match"]
        start = m["end"]

        # Determine where to end this section
        end = len(text)
        for j in range(i + 1, len(matches)):
            if matches[j]["type"] == "article":
                # Only split if it's exactly the next article
                if matches[j]["article_number"] == m.get("article_number", -999) + 1:
                    end = matches[j]["start"]
                    break
            elif matches[j]["type"] == "structure":
                end = matches[j]["start"]
                break

        content = text[start:end].strip()

        if m["type"] == "structure":
            heading_text = match_obj.group("heading").strip()
            for level in hierarchy:
                if heading_text.upper().startswith(level):
                    hierarchy[level] = heading_text
                    # Clear lower hierarchy
                    for k in list(hierarchy.keys())[list(hierarchy).index(level) + 1:]:
                        hierarchy[k] = None
                    break

        elif m["type"] == "article":
            heading_text = match_obj.group("article").strip()
            full_heading = " > ".join([v for v in hierarchy.values() if v]) + f" > {heading_text}"

            sections.append({
                "heading": full_heading.strip(),
                "text": content,
                "sub_heading": [heading_text]
            })

        i += 1

    return sections


In [ ]:
#best_chunk
import re
import numpy as np
from chromadb import PersistentClient
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from sklearn.metrics.pairwise import cosine_similarity
from src.config import EMBEDDING_MODEL_NAME

# -- Hyperparameters --
KEYWORD_BONUS_WEIGHT = 0.05
EXACT_ARTICLE_MATCH_BONUS = 1.0
IMPORTANT_KEYWORDS = [
    "purpose", "goal", "objective", "aim",
    "definition", "refers to", "means",
    "foreign", "breeder", "right", "application",
    "article", "law", "protection", "variety",
    "nsi", "institution", "authority", "registration"
]

def keyword_overlap_score(text: str, query: str) -> int:
    return sum(1 for kw in IMPORTANT_KEYWORDS if kw.lower() in text.lower() and kw.lower() in query.lower())

def extract_article_number_from_query(query: str) -> str:
    match = re.search(r'\barticle\s+(\d+)', query.lower())
    return f"article {match.group(1)}" if match else None

def rerank_chunks(results, query_embedding, query_text):
    documents = results["documents"][0]
    embeddings = np.array(results["embeddings"][0])
    metadatas = results["metadatas"][0]

    query_vec = np.array(query_embedding).reshape(1, -1)
    base_scores = cosine_similarity(query_vec, embeddings)[0]

    article_match = extract_article_number_from_query(query_text)

    reranked = []
    for score, doc, meta in zip(base_scores, documents, metadatas):
        bonus = 0.0

        # Boost for keyword relevance
        bonus += keyword_overlap_score(doc, query_text) * KEYWORD_BONUS_WEIGHT

        # Boost for article number match
        sub_heading = meta.get("sub_heading", "").lower()
        if article_match and article_match in sub_heading:
            bonus += EXACT_ARTICLE_MATCH_BONUS

        final_score = score + bonus
        reranked.append((final_score, score, doc.strip(), meta))

    reranked.sort(key=lambda x: x[0], reverse=True)
    return reranked

def get_top_chunks(country, query, chroma_path, top_k=10):
    print(f"\n Running multilingual chunk search for country: '{country}', query: '{query}'")

    client = PersistentClient(path=chroma_path)
    embedder = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL_NAME)
    collection = client.get_or_create_collection(name="regulations", embedding_function=embedder)

    query_embedding = embedder([query])[0]

    results = collection.query(
        query_texts=[query],
        n_results=top_k * 3,  # Fetch more to allow better reranking
        where={"country": country.lower()},
        include=["documents", "metadatas", "embeddings"]
    )

    if not results.get("documents") or not results["documents"][0]:
        print("No documents found for this country.")
        return []

    reranked = rerank_chunks(results, query_embedding, query)
    return reranked[:top_k]


In [ ]:
#embed_store
import os
import json
from pathlib import Path
from hashlib import sha256

from src.extract import extract_text_from_pdf
from src.chunking import chunk_text
from src.section_splitter import split_into_section
from src.config import EMBEDDING_MODEL_NAME, JSONL_OUTPUT_DIR

from chromadb import PersistentClient
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

from transformers import AutoTokenizer
import tiktoken
from langdetect import detect

# Initialize tokenizer and tiktoken encoder
enc = tiktoken.get_encoding("cl100k_base")
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)
MAX_TOKENS = 512


def count_tokens(text: str) -> int:
    return len(enc.encode(text))


def truncate_text(text: str, max_tokens: int = MAX_TOKENS) -> str:
    input_ids = tokenizer.encode(text, add_special_tokens=False)[:max_tokens]
    return tokenizer.decode(input_ids, skip_special_tokens=True)


def detect_language(text: str) -> str:
    try:
        return detect(text)
    except Exception:
        return "unknown"


def get_fingerprint(text: str, length: int = 50) -> str:
    return sha256(" ".join(text.split()[:length]).encode()).hexdigest()


def process_pdf_to_chroma(
    pdf_path: Path,
    chroma_path: Path,
    collection_name: str = "regulations",
    jsonl_dir: Path = JSONL_OUTPUT_DIR
):
    text = extract_text_from_pdf(pdf_path)
    if not text or not text.strip():
        print(f"No usable text in {pdf_path.name}")
        return

    sections = split_into_section(text)

    # Setup Chroma
    client = PersistentClient(path=chroma_path)
    embedder = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL_NAME)
    collection = client.get_or_create_collection(name=collection_name, embedding_function=embedder)

    # Prepare output
    jsonl_path = jsonl_dir / f"{pdf_path.stem}.jsonl"
    os.makedirs(jsonl_dir, exist_ok=True)

    country = pdf_path.stem.split("_")[0].lower()
    language = detect_language(text)
    seen_fingerprints = set()

    with open(jsonl_path, "w", encoding="utf-8") as f_out:
        for sec_idx, section in enumerate(sections, 1):
            section_heading = section.get("heading", "").strip()
            sub_heading_raw = section.get("sub_heading", [])
            sub_heading = ", ".join(sub_heading_raw) if isinstance(sub_heading_raw, list) else str(sub_heading_raw)

            chunks = chunk_text(section["text"], section_heading=section_heading)

            for chunk_idx, chunk in enumerate(chunks, 1):
                raw_text = f"{section_heading}\n\n{chunk['text']}".strip() if section_heading else chunk['text']
                full_text = truncate_text(raw_text)

                fingerprint = get_fingerprint(full_text)
                if fingerprint in seen_fingerprints:
                    continue
                seen_fingerprints.add(fingerprint)

                embedding = embedder([full_text])[0]
                chunk_id = f"{country}{pdf_path.stem}-s{sec_idx}-c{chunk_idx}"

                metadata = {
                    "chunk_id": chunk_id,
                    "tokens": count_tokens(full_text),
                    "country": country,
                    "language": language,
                    "source_file": pdf_path.name,
                    "section_heading": section_heading,
                    "sub_heading": sub_heading
                }

                # Avoid duplication and store
                collection.delete(ids=[chunk_id])
                collection.add(
                    documents=[full_text],
                    metadatas=[metadata],
                    embeddings=[embedding],
                    ids=[chunk_id]
                )

                # Save to JSONL
                f_out.write(json.dumps({
                    "chunk_id": chunk_id,
                    "text": full_text,
                    "embedding": embedding.tolist(),
                    "metadata": metadata
                }, ensure_ascii=False) + "\n")

    print(f"Processed {pdf_path.name}")
    print(f"Stored in ChromaDB: {chroma_path}")
    print(f"JSONL written to: {jsonl_path}")


def store_to_chroma(entries, chroma_path, collection_name="regulations"):
    client = PersistentClient(path=chroma_path)
    embedder = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL_NAME)
    collection = client.get_or_create_collection(name=collection_name, embedding_function=embedder)

    for entry in entries:
        text = entry["text"]
        metadata = entry["metadata"]
        embedding = entry["embedding"]
        chunk_id = metadata["chunk_id"]

        collection.delete(ids=[chunk_id])
        collection.add(
            documents=[text],
            metadatas=[metadata],
            embeddings=[embedding],
            ids=[chunk_id]
        )


In [ ]:
#extract
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    raw_text = "\n".join(page.get_text() for page in doc)
    # Fix overly spaced words like: "D E C I D E D" -> "DECIDED"
    cleaned_text = re.sub(r'\b(?:[A-Z]\s){2,}[A-Z]\b', lambda m: m.group(0).replace(" ", ""), raw_text)

    return cleaned_text
    # return "\n".join(page.get_text() for page in doc)

In [ ]:
#chunking
from typing import List, Dict
from transformers import AutoTokenizer
import nltk
from nltk.tokenize import sent_tokenize

# nltk.download('punkt')  # Uncomment on first run
tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-large")

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))

def chunk_text(
    section_text: str,
    section_heading: str = "",
    max_tokens: int = 512,
    max_chars: int = 1000,
    sentence_overlap: int = 2
) -> List[Dict]:
    heading_tokens = count_tokens(section_heading)
    token_budget = max_tokens - heading_tokens

    if token_budget <= 0:
        section_heading = ""
        token_budget = max_tokens

    sentences = sent_tokenize(section_text)
    chunks = []
    current_chunk = []
    current_tokens = 0
    current_chars = 0

    for sentence in sentences:
        sentence_tokens = count_tokens(sentence)
        sentence_chars = len(sentence)

        # If sentence is too long, truncate it
        if sentence_tokens > token_budget or sentence_chars > max_chars:
            sentence = tokenizer.decode(tokenizer.encode(sentence)[:token_budget])
            sentence_tokens = count_tokens(sentence)
            sentence_chars = len(sentence)

        if (current_tokens + sentence_tokens > token_budget) or (current_chars + sentence_chars > max_chars):
            if current_chunk:
                chunk_text_combined = " ".join(current_chunk)
                chunks.append({
                    "section_heading": section_heading,
                    "text": chunk_text_combined.strip()
                })
                # Add overlap
                overlap = current_chunk[-sentence_overlap:] if sentence_overlap > 0 else []
                current_chunk = list(overlap)
                current_tokens = sum(count_tokens(s) for s in current_chunk)
                current_chars = sum(len(s) for s in current_chunk)

        current_chunk.append(sentence)
        current_tokens += sentence_tokens
        current_chars += sentence_chars

    if current_chunk:
        chunk_text_combined = " ".join(current_chunk)
        chunks.append({
            "section_heading": section_heading,
            "text": chunk_text_combined.strip()
        })
    return chunks
